# 07 — Frontier-Model Validation: Statistics for Table 3 (GPT-5.5)

This notebook reproduces every statistic reported in **Section 4.5 / Table 3** of the
manuscript *"Toward Practical LLM-assisted Vulnerability Detection: A Specificity-Aware
Ablation Study with Hint-Leakage Controls."*

It is **self-contained**: it reads only the per-sample prediction file
`rq4_frontier_gpt55.csv` produced by notebook `06_run_frontier_validation.ipynb`.
No API key, no network access, and no other repository files are required.

**Inputs**: `rq4_frontier_gpt55.csv` (columns: `File_Name, True_Label, True_CWE,
Variant_A_Baseline, Variant_E_Full`; 234 rows).

**Outputs**: the Variant A vs Variant E performance table (Table 3), the per-CWE recall
breakdown, and two McNemar tests (paired-prediction and correctness-based).


## 1. Setup

Locate the CSV. By default we look for it next to this notebook and in common repository
locations (`results/`, `data/`). Edit `CSV_PATH` if your copy lives elsewhere.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import chi2, binomtest

CSV_NAME = "rq4_frontier_gpt55.csv"

# Search a few sensible locations; override manually if needed.
_candidates = [
    Path(CSV_NAME),
    Path("results") / CSV_NAME,
    Path("data") / CSV_NAME,
    Path("..") / "results" / CSV_NAME,
]
CSV_PATH = next((p for p in _candidates if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        f"Could not find {CSV_NAME}. Set CSV_PATH manually, e.g. "
        f"CSV_PATH = Path('/path/to/{CSV_NAME}')"
    )

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
print(f"Loaded: {CSV_PATH}  ({len(df)} rows)")
print("Columns:", list(df.columns))

## 2. Sanity checks on the benchmark composition

The benchmark is 234 samples: 100 CWE-78 (CmdInj, all Vulnerable), 100 CWE-89 (SQLi, all
Vulnerable), and 34 Safe negatives (no CWE label). We confirm this before computing metrics.

In [ ]:
assert len(df) == 234, f"Expected 234 rows, found {len(df)}"

print("True_Label distribution:")
print(df["True_Label"].value_counts().to_string())
print("\nTrue_CWE distribution (Safe rows carry no CWE):")
print(df["True_CWE"].value_counts(dropna=False).to_string())

# Prediction distributions (reported in the text of Section 4.5).
for col in ["Variant_A_Baseline", "Variant_E_Full"]:
    vc = df[col].str.strip().value_counts().to_dict()
    print(f"\n{col}: {vc}")

## 3. Metric definitions

Positive class = **Vulnerable**. For a set of binary predictions we report Recall
(sensitivity), Specificity, Precision, F1, and Accuracy. On the per-CWE subsets only
Recall is defined, because those subsets contain no Safe (negative) samples; Specificity
is therefore an aggregate quantity computed over the 34 Safe negatives.

In [ ]:
def encode(series):
    """Map the 'Vulnerable'/'Safe' label column to 1/0 (positive = Vulnerable)."""
    return (series.str.strip() == "Vulnerable").astype(int)

y_true = encode(df["True_Label"])
pred_A = encode(df["Variant_A_Baseline"])
pred_E = encode(df["Variant_E_Full"])

def metrics(y, pred):
    TP = int(((pred == 1) & (y == 1)).sum())
    FP = int(((pred == 1) & (y == 0)).sum())
    TN = int(((pred == 0) & (y == 0)).sum())
    FN = int(((pred == 0) & (y == 1)).sum())
    recall = TP / (TP + FN) if (TP + FN) else float("nan")
    spec   = TN / (TN + FP) if (TN + FP) else float("nan")
    prec   = TP / (TP + FP) if (TP + FP) else float("nan")
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) else float("nan")
    acc    = (TP + TN) / len(y)
    return dict(TP=TP, FP=FP, TN=TN, FN=FN,
                Recall=recall, Specificity=spec, Precision=prec, F1=f1, Accuracy=acc)

## 4. Table 3 — Variant A (Baseline) vs Variant E (Full Framework, HINTED)

These are the headline numbers reported in Table 3 of the manuscript.

In [ ]:
mA = metrics(y_true, pred_A)
mE = metrics(y_true, pred_E)

table3 = pd.DataFrame(
    {
        "Variant A (Baseline)": [mA["Accuracy"], mA["F1"], mA["Recall"],
                                 mA["Specificity"], mA["Precision"]],
        "Variant E (Full)":     [mE["Accuracy"], mE["F1"], mE["Recall"],
                                 mE["Specificity"], mE["Precision"]],
    },
    index=["Accuracy", "F1", "Recall", "Specificity", "Precision"],
).round(3)

print("Table 3 — GPT-5.5 (gpt-5.5-2026-04-23), N = 234\n")
print(table3.to_string())
print("\nConfusion-matrix detail:")
print(f"  Variant A: TP={mA['TP']} FP={mA['FP']} TN={mA['TN']} FN={mA['FN']}  "
      f"(Vulnerable verdicts = {mA['TP']+mA['FP']})")
print(f"  Variant E: TP={mE['TP']} FP={mE['FP']} TN={mE['TN']} FN={mE['FN']}  "
      f"(Vulnerable verdicts = {mE['TP']+mE['FP']})")

## 5. Per-CWE recall breakdown

The suppression effect is not uniform across categories. Reported in Section 4.5:
CWE-78 (CmdInj) recall barely moves, whereas CWE-89 (SQLi) recall collapses to zero.

In [ ]:
for cwe, label in [("CWE-78", "CmdInj"), ("CWE-89", "SQLi")]:
    mask = df["True_CWE"].str.strip() == cwe
    rA = metrics(y_true[mask], pred_A[mask])["Recall"]
    rE = metrics(y_true[mask], pred_E[mask])["Recall"]
    n = int(mask.sum())
    print(f"{cwe} ({label}, n={n}):  Recall A = {rA:.3f}  ->  E = {rE:.3f}")

## 6. McNemar tests (Variant A vs Variant E)

We report **two** complementary paired tests, matching the manuscript.

**(a) Paired-prediction test** — the test used throughout the study (Section 3.4). It asks
whether the *predictions* changed asymmetrically between A and E. This is the test behind
the headline χ² = 15.06 (p = 0.0001).

**(b) Correctness-based test** — asks whether *classification correctness* changed
asymmetrically. This is the χ² = 0.94 (p = 0.332) reported as the second qualification in
Section 4.5: the prediction shift is significant and one-directional, but its net effect on
correctness is not, because the suppressed verdicts include both true and false positives.

Both use the continuity-corrected statistic χ² = (|b − c| − 1)² / (b + c); we also report
the exact binomial p-value, which does not rely on the large-sample approximation.

In [ ]:
def mcnemar(flag_a, flag_b, label):
    """flag_* are 1/0 arrays. b = a-only, c = b-only (discordant pairs)."""
    b = int(((flag_a == 1) & (flag_b == 0)).sum())
    c = int(((flag_a == 0) & (flag_b == 1)).sum())
    n = b + c
    print(f"=== {label} ===")
    print(f"  discordant: b = {b}, c = {c}  (n = {n})")
    if n == 0:
        print("  no discordant pairs; test undefined")
        return
    chi_cc = (abs(b - c) - 1) ** 2 / n
    chi_nc = (b - c) ** 2 / n
    p_cc = 1 - chi2.cdf(chi_cc, 1)
    p_nc = 1 - chi2.cdf(chi_nc, 1)
    p_exact = binomtest(min(b, c), n, 0.5).pvalue
    print(f"  chi2 (continuity-corrected) = {chi_cc:.2f}   p = {p_cc:.4f}")
    print(f"  chi2 (uncorrected)          = {chi_nc:.2f}   p = {p_nc:.4f}")
    print(f"  exact binomial p            = {p_exact:.4f}")
    print()

# (a) Paired-prediction: did the verdict (Vulnerable/Safe) change?
#     b = A-Vulnerable -> E-Safe (more conservative); c = A-Safe -> E-Vulnerable.
mcnemar(pred_A, pred_E, "(a) Paired-prediction test  [A=Vuln & E=Safe vs A=Safe & E=Vuln]")

# (b) Correctness: did the prediction become right/wrong?
correct_A = (pred_A == y_true).astype(int)
correct_E = (pred_E == y_true).astype(int)
mcnemar(correct_A, correct_E, "(b) Correctness-based test  [A-correct/E-wrong vs A-wrong/E-correct]")

## 7. Direction of the prediction shift

The paired-prediction test above is significant *and* perfectly one-directional. Here we
confirm that every changed verdict moved Vulnerable → Safe (none reversed), and show where
those suppressed verdicts land by true label and CWE.

In [ ]:
shift_mask = (pred_A == 1) & (pred_E == 0)   # Vulnerable -> Safe
reverse_mask = (pred_A == 0) & (pred_E == 1)  # Safe -> Vulnerable

print(f"Vulnerable -> Safe shifts: {int(shift_mask.sum())}")
print(f"Safe -> Vulnerable shifts: {int(reverse_mask.sum())}")
print()
print("Among the Vulnerable -> Safe shifts, true labels:")
print(df.loc[shift_mask, "True_Label"].value_counts().to_string())
print("\nAmong the Vulnerable -> Safe shifts, by CWE:")
print(df.loc[shift_mask, "True_CWE"].value_counts(dropna=False).to_string())

## 8. Summary

Running this notebook on `rq4_frontier_gpt55.csv` reproduces:

- **Table 3**: Variant A (Acc 0.308, F1 0.367, Recall 0.235, Spec 0.735, Prec 0.839) vs
  Variant E (Acc 0.286, F1 0.301, Recall 0.180, Spec 0.912, Prec 0.923).
- **Per-CWE recall**: CWE-78 0.390 → 0.360; CWE-89 0.080 → 0.000.
- **Paired-prediction McNemar**: χ² = 15.06, p = 0.0001 (17 discordant verdicts, all
  Vulnerable → Safe).
- **Correctness McNemar**: χ² = 0.94, p = 0.332 (11 vs 6 discordant; not significant).

These correspond to the values reported in Section 4.5 of the manuscript.
